In [ ]:
# ── Cell 1 of Global_Server.ipynb ────────────────────────────
#
# PORTABLE sys.path BOOTSTRAP
# Works regardless of where the repo lives on any machine.
# __file__ is not available in Jupyter, so we use the known
# relative location: this notebook is at server/Global_Server.ipynb
# → project root is one level up.
#
import sys, os

# Resolve project root from the notebook's own directory
_HERE        = os.path.abspath(os.getcwd())          # .../project/server
_PROJECT_ROOT = os.path.dirname(_HERE)               # .../project
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

import pickle, ssl
from flask import Flask, request
import tenseal as ts

from config import (
    RESULTS_PATH, SERVER_PORT,
    CERT_FILE, KEY_FILE, CHUNKS_FOLDER
)

# All paths now come from config — no strings needed here
CONTEXT_FILE = os.path.join(RESULTS_PATH, "context.ser")
os.makedirs(CHUNKS_FOLDER, exist_ok=True)

# ── Load CKKS Context ─────────────────────────────────────────
if not os.path.exists(CONTEXT_FILE):
    print(f"[SERVER] ERROR: context.ser not found at {CONTEXT_FILE}")
    print("[SERVER] Run client Cell 10 first to generate it.")
    client_context = None
else:
    with open(CONTEXT_FILE, "rb") as f:
        client_context = ts.context_from(f.read())
    print(f"[SERVER] CKKS context loaded from {CONTEXT_FILE}")

# ── Flask App ─────────────────────────────────────────────────
app = Flask(__name__)

@app.route("/", methods=["POST"])
def receive_chunk():
    try:
        payload   = pickle.loads(request.data)
        chunk_id  = payload["chunk_id"]
        client_id = payload["client_id"]
        data      = payload["data"]

        # Validate CKKS ciphertext — reject tampered or malformed data
        try:
            encrypted = ts.ckks_vector_from(client_context, data)
            if encrypted.size() < 5 or encrypted.size() > 10000:
                print(f"[SERVER] Dropped chunk {chunk_id} — "
                      f"invalid size: {encrypted.size()}")
                return "Corrupted", 400
        except Exception as e:
            print(f"[SERVER] Dropped chunk {chunk_id} from "
                  f"{client_id} — CKKS validation failed: {e}")
            return "Corrupted", 400

        # Save validated chunk
        client_folder = os.path.join(CHUNKS_FOLDER, client_id)
        os.makedirs(client_folder, exist_ok=True)
        filename = os.path.join(client_folder, f"chunk_{chunk_id}.bin")
        with open(filename, "wb") as f:
            pickle.dump({"data": data}, f)

        print(f"[SERVER] ✓ Received chunk {chunk_id} from {client_id}")
        return "OK", 200

    except Exception as e:
        print(f"[SERVER] Error: {e}")
        return "Error", 500

@app.route("/health", methods=["GET"])
def health():
    return "OK", 200

# ── Start Server ──────────────────────────────────────────────
def run_server():
    if not os.path.exists(CERT_FILE):
        print(f"[SERVER] ERROR: cert.pem not found at {CERT_FILE}")
        print("[SERVER] Generate certs once with:")
        print(f"  cd {os.path.dirname(CERT_FILE)}")
        print("  openssl req -x509 -newkey rsa:2048 -keyout key.pem "
              "-out cert.pem -days 365 -nodes -subj '/CN=127.0.0.1'")
        return

    ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
    ssl_context.load_cert_chain(certfile=CERT_FILE, keyfile=KEY_FILE)

    print(f"[SERVER] Chunks folder : {CHUNKS_FOLDER}")
    print(f"[SERVER] Listening securely on https://127.0.0.1:{SERVER_PORT} ...")
    app.run(
        host="127.0.0.1",
        port=SERVER_PORT,
        ssl_context=ssl_context,
        threaded=True,
    )

run_server()

In [ ]:
import os
print("cert.pem exists:", os.path.exists("cert.pem"))
print("key.pem exists:", os.path.exists("key.pem"))


In [ ]:
file_path = "received_gradients.json"

with open(file_path, "r", encoding="utf-8") as f:
    raw = f.read()

print("Raw length:", len(raw))
print("Preview:\n", raw[:1000])  # First 1000 characters


In [ ]:
with open("fixed_received_gradients.json", "r") as f:
    data = json.load(f)

print(f"Keys: {list(data.keys())}")
print(f"Number of chunks: {len(data['all_encrypted_gradients'])}")
print(f"First 100 chars of chunk 0: {data['all_encrypted_gradients'][0][:100]}")



In [ ]:
import base64
import tenseal as ts

with open("context.ser", "rb") as f:
    context = ts.context_from(f.read())

# Get the first chunk
chunk = data['all_encrypted_gradients'][0]

try:
    decoded = base64.b64decode(chunk)
    vec = ts.ckks_vector_from(context, decoded)
    print("Successfully deserialized chunk 0.")
except Exception as e:
    print("Still not valid CKKS vector:", e)


In [ ]:
import ssl
print("SSL module path:", ssl.__file__)


In [ ]:
import os
print("File exists:", os.path.exists("received_gradients.json"))